In [1]:
import pandas as pd 
import numpy as np

df=pd.read_csv('btcusd_1-min_data.csv')


In [2]:
#give max date and min date
print(df['Timestamp'].max())
print(df['Timestamp'].min())

1764547140.0
1325412060.0


In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import datetime
import config

CSV_PATH = "E:\\ML_AI\\data\\btcusd_1-min_data.csv"     # your CSV file
TICKER = "BTCUSD"              # change per file or pass as argument

# Convert cutoff timestamp (2025-10-11 11:02:00)
CUTOFF_UNIX = 1760180520

# PostgreSQL connection
DB_URL = (
    f"postgresql+psycopg2://{config.DB_USER}:{config.DB_PASS}"
    f"@{config.DB_HOST_OUTSIDE}:5432/{config.DB_NAME}"
)
engine = create_engine(DB_URL)


def unix_to_timestamp(x):
    return datetime.datetime.utcfromtimestamp(int(x))


def ingest_csv():
    print(f"📥 Loading CSV: {CSV_PATH}")
    df = pd.read_csv(CSV_PATH)

    # Convert column names (optional)
    df.columns = ["timestamp", "open", "high", "low", "close", "volume"]

    # Filter only new rows
    df_new = df[df["timestamp"] > CUTOFF_UNIX].copy()

    if df_new.empty:
        print("ℹ No new rows to insert.")
        return

    # Convert UNIX → timestamp
    df_new["date"] = df_new["timestamp"].apply(unix_to_timestamp)

    # Add ticker column
    df_new["ticker"] = TICKER

    # Select final columns
    df_final = df_new[["date", "open", "high", "low", "close", "volume", "ticker"]]

    print(f"📝 Inserting {len(df_final)} new rows into PostgreSQL...")

    df_final.to_sql(
        "crypto_market_data",
        engine,
        if_exists="append",
        index=False,
        method="multi",
        chunksize=5000,
    )

    print("🎉 Done! New rows inserted.")


if __name__ == "__main__":
    ingest_csv()


📥 Loading CSV: E:\ML_AI\data\btcusd_1-min_data.csv
📝 Inserting 72777 new rows into PostgreSQL...
🎉 Done! New rows inserted.


In [18]:
import pandas as pd
import datetime
from sqlalchemy import create_engine
import config

# ------------------------
# CONFIG
# ------------------------
CSV_FILE = "E:\\ML_AI\\data\\Binance_BNBUSDT_2025_minute.csv"
TICKER = "BNBUSD"   # Convert USDT → USD (label only)

# Cutoff timestamp (DB starts at this time)
CUTOFF_TS = "2025-10-11 11:03:00.000000"
CUTOFF_DT = datetime.datetime.fromisoformat(CUTOFF_TS)

# PostgreSQL
DB_URL = (
    f"postgresql+psycopg2://{config.DB_USER}:{config.DB_PASS}"
    f"@{config.DB_HOST_OUTSIDE}:5432/{config.DB_NAME}"
)
engine = create_engine(DB_URL)


def ingest_ada():

    print(f"📥 Loading CSV with skiprows=1: {CSV_FILE}")

    # IMPORTANT: skip the first junk row
    df = pd.read_csv(CSV_FILE, skiprows=1)

    # Normalize column names
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    # Expected now:
    # ['date','unix','symbol','open','high','low','close','volume_ada','volume_usdt','tradecount']

    # Convert date to datetime
    df["date"] = pd.to_datetime(df["date"])

    # Filter for new rows only
    df_new = df[df["date"] > CUTOFF_DT].copy()

    if df_new.empty:
        print("ℹ No new ADA rows to insert.")
        return

    # Create final DataFrame for DB
    df_final = pd.DataFrame({
        "date": df_new["date"],
        "open": df_new["open"],
        "high": df_new["high"],
        "low": df_new["low"],
        "close": df_new["close"],
        "volume": df_new["volume_bnb"],   # use ADA volume
        "ticker": TICKER
    })

    print(f"📝 Inserting {len(df_final)} ADA rows into PostgreSQL...")

    df_final.to_sql(
        "crypto_market_data",
        engine,
        if_exists="append",
        index=False,
        method="multi",
        chunksize=5000
    )

    print("🎉 BNB ingestion complete!")


# ------------------------
# RUN
# ------------------------
if __name__ == "__main__":
    ingest_ada()


📥 Loading CSV with skiprows=1: E:\ML_AI\data\Binance_BNBUSDT_2025_minute.csv
📝 Inserting 71334 ADA rows into PostgreSQL...
🎉 BNB ingestion complete!


In [ ]:
import pandas as pd
import requests
from sqlalchemy import create_engine, text
from datetime import datetime, timedelta, timezone
import config

# -----------------------------
# DB CONNECTION
# -----------------------------
DB_URL = (
    f"postgresql+psycopg2://{config.DB_USER}:{config.DB_PASS}"
    f"@{config.DB_HOST_OUTSIDE}:5432/{config.DB_NAME}"
)

engine = create_engine(DB_URL)

# -----------------------------
# SYMBOL MAP
# -----------------------------
COINBASE = {
    "BTCUSD": "BTC-USD",
    "ETHUSD": "ETH-USD",
    "ADAUSD": "ADA-USD",
    "XRPUSD": "XRP-USD",
}

TABLE = "check_crypto"


# -----------------------------
#  GET LATEST TIMESTAMP FROM DB
# -----------------------------
def get_latest_dt(ticker):
    q = text(f"""
        SELECT MAX(date)
        FROM {TABLE}
        WHERE ticker = :t
    """)
    with engine.connect() as conn:
        result = conn.execute(q, {"t": ticker}).scalar()
    return result  # naive timestamp


# -----------------------------
# FETCH FROM COINBASE
# -----------------------------
def fetch_coinbase(symbol, start_dt):
    end_dt = datetime.now(timezone.utc)

    url = (
        f"https://api.exchange.coinbase.com/products/{symbol}/candles"
        f"?granularity=60"
        f"&start={start_dt.isoformat()}"
        f"&end={end_dt.isoformat()}"
    )

    r = requests.get(url)

    if r.status_code != 200:
        print("❌ Coinbase error:", r.text)
        return None

    data = r.json()

    if not isinstance(data, list):
        print("❌ Unexpected Coinbase response:", data)
        return None

    # Coinbase returns newest-first → reverse
    data.reverse()

    df = pd.DataFrame(data, columns=[
        "time", "low", "high", "open", "close", "volume"
    ])

    df["date"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df.drop(columns=["time"], inplace=True)

    return df


# -----------------------------
# INGEST LOGIC
# -----------------------------
def ingest(ticker):
    print(f"\n🚀 Fetching last 24h for {ticker} ...")

    cb_symbol = COINBASE[ticker]

    latest = get_latest_dt(ticker)

    if latest is None:
        # fallback: fetch full last 24 hours
        start_dt = datetime.now(timezone.utc) - timedelta(days=1)
    else:
        # Coinbase requires timezone-aware UTC
        start_dt = latest.replace(tzinfo=timezone.utc)

    df = fetch_coinbase(cb_symbol, start_dt)
    if df is None or df.empty:
        print("ℹ No data returned.")
        return

    # Only keep rows > latest
    if latest:
        df = df[df["date"] > latest]

    if df.empty:
        print("ℹ Nothing new to insert.")
        return

    df["ticker"] = ticker
    df.rename(columns={"volume": "volume"}, inplace=True)

    # -------------------------
    # Insert
    # -------------------------
    with engine.begin() as conn:
        df.to_sql(TABLE, conn, if_exists="append", index=False)

    print(f"✔ Inserted {len(df)} rows for {ticker}.")


# -----------------------------
# MAIN
# -----------------------------
if __name__ == "__main__":
    print("🚀 Starting 24-hour ingestion...\n")

    for t in COINBASE:
        ingest(t)

    print("\n🎉 Incremental ingestion complete!")


🚀 Starting 24-hour ingestion...

🚀 Fetching last 24h for BTCUSD...
❌ Binance error: {
  "code": 0,
  "msg": "Service unavailable from a restricted location according to 'b. Eligibility' in https://www.binance.com/en/terms. Please contact customer service if you believe you received this message in error."
}
ℹ No new data returned.

🚀 Fetching last 24h for ETHUSD...
❌ Binance error: {
  "code": 0,
  "msg": "Service unavailable from a restricted location according to 'b. Eligibility' in https://www.binance.com/en/terms. Please contact customer service if you believe you received this message in error."
}
ℹ No new data returned.

🚀 Fetching last 24h for ADAUSD...
❌ Binance error: {
  "code": 0,
  "msg": "Service unavailable from a restricted location according to 'b. Eligibility' in https://www.binance.com/en/terms. Please contact customer service if you believe you received this message in error."
}
ℹ No new data returned.

🚀 Fetching last 24h for XRPUSD...
❌ Binance error: {
  "code": 0

In [ ]:
import pandas as pd
import requests
from sqlalchemy import create_engine, text
from datetime import datetime, timedelta, timezone
import config

# -----------------------------
# DB CONNECTION
# -----------------------------
DB_URL = (
    f"postgresql+psycopg2://{config.DB_USER}:{config.DB_PASS}"
    f"@{config.DB_HOST_OUTSIDE}:5432/{config.DB_NAME}"
)

engine = create_engine(DB_URL)

# -----------------------------
# SYMBOL MAP
# -----------------------------
COINBASE = {
    "BTCUSD": "BTC-USD",
    "ETHUSD": "ETH-USD",
    "ADAUSD": "ADA-USD",
    "XRPUSD": "XRP-USD",
}

TABLE = "check_crypto"


# -----------------------------
#  GET LATEST TIMESTAMP
# -----------------------------
def get_latest_dt(ticker):
    q = text(f"""
        SELECT MAX(date)
        FROM {TABLE}
        WHERE ticker = :t
    """)
    with engine.connect() as conn:
        result = conn.execute(q, {"t": ticker}).scalar()
    return result  # UTC-naive from DB


# -----------------------------
# FETCH CHUNK FROM COINBASE (≤300)
# -----------------------------
def fetch_chunk(symbol, start_dt, end_dt):
    url = (
        f"https://api.exchange.coinbase.com/products/{symbol}/candles"
        f"?granularity=60"
        f"&start={start_dt.isoformat()}"
        f"&end={end_dt.isoformat()}"
    )

    r = requests.get(url)
    if r.status_code != 200:
        print("❌ Coinbase error:", r.text)
        return None

    data = r.json()
    if not isinstance(data, list):
        print("❌ Unexpected response:", data)
        return None

    # Coinbase sends newest -> oldest
    data.reverse()

    return pd.DataFrame(data, columns=[
        "time", "low", "high", "open", "close", "volume"
    ])


# -----------------------------
# MAIN INGESTION USING CHUNKS
# -----------------------------
def ingest(ticker):
    print(f"\n🚀 Fetching last 24h for {ticker} ...")

    cb_symbol = COINBASE[ticker]
    now = datetime.now(timezone.utc)

    target_start = now - timedelta(hours=24)

    # Get DB last timestamp
    latest = get_latest_dt(ticker)

    if latest:
        # use whichever is later: your DB timestamp OR 24h ago
        target_start = max(latest.replace(tzinfo=timezone.utc), target_start)

    print(f"⏳ Need data from {target_start} → {now}")

    all_rows = []

    # Loop chunks backward
    chunk_end = now
    chunk_size = timedelta(minutes=300)

    while chunk_end > target_start:
        chunk_start = max(target_start, chunk_end - chunk_size)

        df = fetch_chunk(cb_symbol, chunk_start, chunk_end)
        if df is None or df.empty:
            break

        all_rows.append(df)

        # Move window backward
        chunk_end = chunk_start

    if not all_rows:
        print("ℹ No new data.")
        return

    df = pd.concat(all_rows)
    df["date"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df.drop(columns=["time"], inplace=True)

    # Only new rows
    if latest:
        df = df[df["date"] > latest.replace(tzinfo=timezone.utc)]

    if df.empty:
        print("ℹ Nothing new to insert.")
        return

    # Add ticker
    df["ticker"] = ticker

    # Insert
    with engine.begin() as conn:
        df.to_sql(TABLE, conn, if_exists="append", index=False)

    print(f"✔ Inserted {len(df)} rows for {ticker}.")


# -----------------------------
# MAIN
# -----------------------------
if __name__ == "__main__":
    print("🚀 Starting 24-hour ingestion...\n")

    for t in COINBASE:
        ingest(t)

    print("\n🎉 Incremental ingestion complete!")

if __name__ == "__main__":
    print("🚀 Starting 24-hour ingestion...\n")

    for t in COINBASE:
        ingest(t)

    print("\n🎉 Incremental ingestion complete!")